# Spellchecker quest

Хтось наробив помилок у віршах Тараса Шевченка. Наша задача -- виправити ці помилки і прочитати приховане повідомлення.

## Задача

Ви отримаєте тренувальні та тестувальні дані.

Тренувальні дані знаходяться в полі `lab.train_text`. Це звичайний нерозмічений текст. На ньому необхідно натренувати мовну модель. Підійде будь-яка. Я би радив feed-forward нейрону модель з токенізацією по літерах, бо це те, що ми проходили на останній лекції. Але n-грамна теж має спрацювати.

Тестувальні дані знаходиться в полі `lab.test_items`. Приклад одного елемента:

```json
{
  "text": "Співали б прозу, та по ножах,",
  "error_start": 23,
  "error_end": 28,
  "error": "ножах",
  "corrections": [
    "ногах",
    "йотах",
    "єнотах",
    "ножах",
    "нотах"
  ]
}
```

`error_start` та `error_end` вказують на місцезнаходження помилки в тексті (в символах). У данному прикладі, помилкою є `text[23:28]`, тобто слово "ножах".

`corrections` -- це список можливих виправлень.

Ваша задача -- обрати правильне виправлення серед запропонованих.


## Приховане повідомлення

Один приклад в `lab.test_items` дає можливість прочитати одну літеру прихованого повідомлення. Для цього знайдіть різницю між літерами слова з помилкою (`error`) та обраним виправленням. Надрукуйте цю літеру. Якщо слово з помилкою направді правильне, а таке теж буває, надрукуйте пробіл. Приклади:

```
Error               Correction     To print
-------------------------------------------
привіт               приліт        л
пні                  поні          о
баллет               балет         л
привіт               привіт        (space)
```

Приховане повідомлення, яке ви побачите в результаті це рядок з віршу одного з українських авторів.

Відповідь на квест -- ім'я автора/ки у форматі "Ім'я Прізвище".

Полетіли! 🚀

In [1]:
!pip install --quiet --ignore-installed http://nlp.band/static/pypy/lpnlp-2023.10.2-py3-none-any.whl

In [2]:
import lpnlp

lab = lpnlp.start(
    email="oleh.lozovyi.kn.2021@lpnu.ua",                   # <----------- Заповніть це поле
    lab="quest_spellchecker"
    )

Удачі!


## Мовна модель

Натренуйте свою мовну модель тут

In [3]:
print(lab.train_text[:330])

﻿ПРИЧИННА

Реве та стогне Дніпр широкий,
Сердитий вітер завива,
Додолу верби гне високі,
Горами хвилю підійма.
І блідий місяць на ту пору
Із хмари де-де виглядав,
Неначе човен в синім морі,
То виринав, то потопав.
Ще треті півні не співали,
Ніхто нігде не гомонів,
Сичі в гаю перекликались,
Та ясен раз у раз скрипів.


In [4]:
import math
import collections
from typing import List, Tuple

import string

def build_vocab_and_model(train_text: str, n: int = 3) -> Tuple[collections.Counter, dict]:

    tokens = list(train_text)

    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

    model = collections.Counter(ngrams)
    vocab = collections.Counter(tokens)

    return vocab, model

vocab, model = build_vocab_and_model(lab.train_text, n=3)

# Читаємо між рядків

In [5]:
# Допоміжна фунція:
def get_letter(w1: str, w2: str) -> str:
    """Повертає літеру, якої відрізняються слова або пробіл для однакових слів.
    """

    letters1 = collections.Counter(w1)
    letters2 = collections.Counter(w2)

    diff = letters1 - letters2
    if len(diff) != 1:
        return " "

    return diff.most_common()[0][0]

def score_text(text: str, model, vocab) -> float:
    """Повертає perplexity або log-probability для тексту. """
    n = 3
    tokens = list(text)
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

    log_prob = 0.
    for ngram in ngrams:
        count_ngram = model.get(ngram, 1)
        count_prefix = sum(v for key, v in model.items() if key[:-1] == ngram[:-1])
        prob = count_ngram / max(count_prefix, 1)
        log_prob += math.log(prob)

    # Обчислюємо перплексію
    l = len(ngrams)
    perplexity = math.exp(-log_prob / l) if l > 0 else float('inf')

    ####################################################
    return perplexity
    ####################################################


# Можете змінювати параметри та весь цей код, якщо потрібно
def solve(model, vocab, test_items) -> Tuple[List[str], str]:
    """Повертає список виправлених слів та секретне повідомлення."""
    choices = []
    secret = []

    for item in test_items:
        scores = []
        for corr in item['corrections']:
            # Підставляємо виправлення в текст
            text = item['text'][:item['error_start']] + corr + item['text'][item['error_end']:]

            # Рахуємо перплексію тексту
            score = score_text(text, model, vocab)
            scores.append(score)

        # Обираємо слово з найменшою перплексією
        result = sorted(zip(scores, item['corrections']), key=lambda x: x[0])
        best_word = result[0][1]
        choices.append(best_word)

        # Знаходимо літеру для секретного повідомлення
        error = item['error']
        letter = get_letter(error, best_word)
        secret.append(letter)

    secret_message = ''.join(secret)
    return choices, secret_message

choices, secret_message = solve(model, vocab, lab.test_items)

lab.evaluate_accuracy(choices)
print("SECRET MESSAGE: ", secret_message)


Відповідь правильна ✅
accuracy = 0.52. Вже краще. Можеш попрацювати над моделлю ще, а можеш рухатися далі й спробувати розгадати приховане повідомлення
SECRET MESSAGE:    йнп   да  р л   м   унТус    реба землі н маєтт   уде  е ожнемаєспоЛ  то буД юво   н м єжпариюто буд тьр Мари  іць  у нап  но пр в а пт  и а а    же люд нанам  Мж л дин   и еннагзе лі са а не літаЄ аш рилажмає а кри  м а   о ив і к и а   с  пУ уп р'    з праВ  т ес оти і д в  'я утк г м з в р о  і   к х   і унко   ез   чн го п   ва н    к г  о  щиро  і    робо и у к го    щед о ті    турб тиХутког   з Пісн чабо   Н д їтабо     ез ї  б з  мрі злюДин   ібито не  іта  аткрил  м є а   ида м Є


In [6]:
lab.answer("Ліна Костенко")

Відповідь правильна ✅
Правильно! 🚀 Заповни тепер цю форму, будь ласка: https://tally.so/r/wkl0zZ


Відправте посилання на цей colab або PDF з ним на пошту oleksii.o.syvokon@lpnu.ua. Дякую!
